# Labo 38 · Carnet Colab — Un modèle ouvert sur GPU

Un petit modèle de langage ouvert, **Qwen2.5-0.5B-Instruct** (un demi-milliard de paramètres), tourne sur le GPU de Colab. Il est beaucoup moins capable que les grands assistants, et c'est instructif : ses erreurs se voient.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=False)
import json, re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
nom = "Qwen/Qwen2.5-0.5B-Instruct"
decoupeur = AutoTokenizer.from_pretrained(nom)
modele = AutoModelForCausalLM.from_pretrained(nom, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
modele = modele.to("cuda" if torch.cuda.is_available() else "cpu")
print(sum(p.numel() for p in modele.parameters()) / 1e6, "millions de paramètres")

def discuter(messages, temperature=0.7, max_jetons=200):
    texte = decoupeur.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    entree = decoupeur(texte, return_tensors="pt").to(modele.device)
    options = dict(max_new_tokens=max_jetons, do_sample=temperature > 0)
    if temperature > 0:
        options["temperature"] = temperature
    sortie = modele.generate(**entree, **options)
    return decoupeur.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True)

## 1. Les messages et la température

**À toi** : pose la même question (« Propose un nom pour un club de robotique au cégep. ») trois fois avec T = 0.2, puis trois fois avec T = 1.2. Compare la variété des réponses.

In [ ]:
# À toi

## 2. Une requête brute contre une requête construite

**À toi** : demande un résumé du texte ci-dessous, une fois avec une requête brute (« Résume. »), une fois avec une requête construite selon le cours : un message système (rôle, public), le texte entre délimiteurs, une longueur et un format précis. Compare.

In [ ]:
texte = ("""La session d'automne commence à la fin du mois d'août. L'étudiant peut abandonner un cours sans mention d'échec
jusqu'au 20 septembre. Au-delà de 10 % d'absences non motivées, l'enseignant peut refuser l'accès à l'évaluation finale.
Un travail remis en retard perd 10 % de sa valeur par jour, jusqu'à un maximum de 5 jours.""")
# À toi

## 3. Une sortie JSON, validée

**À toi** : demande au modèle de classer des messages d'étudiants en JSON (`{"categorie": "absence" | "note" | "inscription" | "autre", "urgence": 1 à 5}`), avec un exemple dans la requête. Fais 10 essais (T = 0.7) et compte combien de réponses sont du JSON valide avec les bonnes clés (`json.loads` dans un `try`).

In [ ]:
messages_etudiants = ["Je serai absent demain pour un rendez-vous médical.", "Ma note de l'examen 2 est-elle publiée?",
                      "Comment je m'inscris au cours d'été?", "Le distributeur à café est brisé."]
# À toi

## 4. Le RAG de l'atelier, avec un vrai modèle

**À toi** : reprends le règlement et la fonction `construire` de l'atelier (recopie-les), retrouve les deux meilleurs extraits par TF-IDF, et demande au modèle de répondre (T = 0.2) à deux questions. La réponse cite-t-elle le bon article? Pose aussi une question dont la réponse n'est **pas** dans le règlement : le modèle le dit-il?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
reglement = [
    "Article 2. Absences. Au-delà de 10 % d'absences non motivées dans un cours, l'enseignant peut refuser l'accès à l'évaluation finale de ce cours.",
    "Article 3. Absence à une évaluation. L'étudiant absent à une évaluation doit fournir une pièce justificative dans les 5 jours ouvrables pour obtenir une reprise.",
    "Article 4. Retards dans les travaux. Un travail remis en retard perd 10 % de sa valeur par jour, jusqu'à un maximum de 5 jours, après quoi il reçoit la note zéro.",
    "Article 8. Note de passage. La note de passage d'un cours est 60 %.",
]
# À toi

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : deux réponses à la même question avec T = 0.2 et T = 1.2, ta requête construite et le résumé obtenu, la proportion de sorties JSON valides, et les réponses du RAG à deux questions (avec l'article cité), plus sa réponse à la question hors règlement.